# Sentinel · reranker fine-tune v2: teacher denoising + distillation

v1 (labels only) made the reranker worse on real questions: the synthetic labels are noisy. v2 has a stronger cross-encoder, [BAAI/bge-reranker-base](https://huggingface.co/BAAI/bge-reranker-base) (MIT, pinned), score every training candidate; training then drops what the teacher contradicts and learns from its scores (distillation).

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about an hour; keep the tab open).
2. The last cell downloads `sentinel-reranker-v2.zip` — unzip it into the repo root.

Hyperparameters were fixed before any ground-truth result was seen: lr 1e-5 (v1's choice), 2 epochs, distillation weight 0.5, temperature 1, teacher top-3.

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!test -d Sentinel || git clone --depth 1 --branch feat/reranker-finetune https://github.com/krishivsaini/Sentinel.git
%cd Sentinel
!git log --oneline -1

In [ ]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

**1. Teacher scores** — every (question, candidate) pair, ~441k; resumable if the runtime drops.

In [ ]:
!python scripts/teacher_scores.py --out data/reranker/teacher.jsonl.gz --device cuda

In [ ]:
import gzip, json, statistics
rows = [json.loads(l) for l in gzip.open('data/reranker/teacher.jsonl.gz', 'rt')]
ranks = [1 + sum(x > r['scores'][0] for x in r['scores'][1:]) for r in rows]
print(len(rows), 'questions | teacher ranks the labelled positive 1st in %.0f%%, top-3 in %.0f%%, median rank %s'
      % (100 * sum(r == 1 for r in ranks) / len(ranks), 100 * sum(r <= 3 for r in ranks) / len(ranks), statistics.median(ranks)))

**2. Check the loop on the GPU** — seconds.

In [ ]:
!python scripts/train_reranker.py --smoke --device cuda --teacher-path data/reranker/teacher.jsonl.gz --out-dir runs/v2-smoke

**3. v2** — denoised data, distillation from the teacher.

In [ ]:
!python scripts/train_reranker.py --lr 1e-5 --epochs 2 --eval-every 400 --limit-val 800 --teacher-path data/reranker/teacher.jsonl.gz --out-dir runs/v2-main

**4. v2, strict** — the same, with every ground-truth passage excluded from training.

In [ ]:
!python scripts/train_reranker.py --lr 1e-5 --epochs 2 --eval-every 400 --limit-val 800 --teacher-path data/reranker/teacher.jsonl.gz --exclude-gold --out-dir runs/v2-nogold

In [ ]:
import shutil, os
os.makedirs('export/runs', exist_ok=True)
for f in ['data/reranker/teacher.jsonl.gz', 'data/reranker/teacher.jsonl.gz.meta.json']:
    os.makedirs(os.path.dirname('export/' + f), exist_ok=True); shutil.copy(f, 'export/' + f)
for r in ['v2-smoke', 'v2-main', 'v2-nogold']:
    os.makedirs(f'export/runs/{r}', exist_ok=True)
    for f in ['log.jsonl', 'summary.json']:
        if os.path.exists(f'runs/{r}/{f}'): shutil.copy(f'runs/{r}/{f}', f'export/runs/{r}/{f}')
for r in ['v2-main', 'v2-nogold']:
    shutil.copytree(f'runs/{r}/best', f'export/runs/{r}/best')
shutil.make_archive('sentinel-reranker-v2', 'zip', 'export')
print(os.path.getsize('sentinel-reranker-v2.zip') // 2**20, 'MB')

In [ ]:
from google.colab import files
files.download('sentinel-reranker-v2.zip')